<a href="https://colab.research.google.com/github/yugan243/Deep-Learning-Pytorch/blob/main/9_simple_mnist_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torch.optim import optimizer
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import csv



In [ ]:
train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5], std=[0.5])
        ])
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5], std=[0.5])
        ])
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 61.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.74MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.6MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.34MB/s]


In [ ]:
print(f"train data size: {train_data.data.shape}")
print(f"sample train data: {train_data[0]}")
print(f"test data size: {test_data.data.shape}")
print(f"sample test data: {test_data[0]}")



train data size: torch.Size([60000, 28, 28])
sample train data: (Image([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

In [ ]:
train_dataloader  = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True,
    pin_memory=True
    )

test_dataloader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False,
    pin_memory=True
    )

In [ ]:
print(next(iter(train_dataloader))[0].shape)

torch.Size([64, 1, 28, 28])


In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=0),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=0),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(1600, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)



"""
Parameter count breakdown — MNISTClassifier
---------------------------------------------
conv1   (1  -> 16,  k=3, p=0):   1*16*3*3 + 16        =     160
conv2   (16 -> 32,  k=3, p=0):  16*32*3*3 + 32        =   4,640
conv3   (32 -> 64,  k=3, p=0):  32*64*3*3 + 64        =  18,496
bn1     (16 channels):          16*2                  =      32
bn2     (32 channels):          32*2                  =      64
bn3     (64 channels):          64*2                  =     128
linear1 (1600 -> 128):        1600*128 + 128          = 204,928
bn4     (128 features):        128*2                  =     256
linear2 (128 -> 10):            128*10 + 10           =   1,290
---------------------------------------------
Total                                                  ≈ 229,994

Reference point: LeNet-5 (1998, the original MNIST-purpose CNN)
uses ~60k params and reaches ~99% test accuracy. This model has
~3.8x that, well within a safe range for a 60,000-image training
set (params roughly same order of magnitude as sample count keeps
overfitting risk low). ~89% of total params sit in linear1 alone,
since Linear layers scale with flattened_size x out_features while
Conv layers stay
"""

'\nParameter count breakdown — MNISTClassifier\n---------------------------------------------\nconv1   (1  -> 16,  k=3, p=0):   1*16*3*3 + 16        =     160\nconv2   (16 -> 32,  k=3, p=0):  16*32*3*3 + 32        =   4,640\nconv3   (32 -> 64,  k=3, p=0):  32*64*3*3 + 64        =  18,496\nbn1     (16 channels):          16*2                  =      32\nbn2     (32 channels):          32*2                  =      64\nbn3     (64 channels):          64*2                  =     128\nlinear1 (1600 -> 128):        1600*128 + 128          = 204,928\nbn4     (128 features):        128*2                  =     256\nlinear2 (128 -> 10):            128*10 + 10           =   1,290\n---------------------------------------------\nTotal                                                  ≈ 229,994\n\nReference point: LeNet-5 (1998, the original MNIST-purpose CNN)\nuses ~60k params and reaches ~99% test accuracy. This model has\n~3.8x that, well within a safe range for a 60,000-image training\nset (para

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(device)

cuda


In [ ]:
model = MNISTClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


In [ ]:
epochs = 15
loss_history = []
acc_history = []

In [ ]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    loss_history.append(epoch_loss)
    acc_history.append(epoch_acc)

    print(f"Epoch {epoch+1}/{epochs}: loss={epoch_loss:.4f}, train_acc={epoch_acc:.6f}")

KeyboardInterrupt: 

In [ ]:

filepath = '/content/drive/MyDrive/GAN_Assignment/classifier_training_log.csv'
os.makedirs(os.path.dirname(filepath), exist_ok=True)

with open(filepath, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'train_loss', 'train_acc'])
    for i, (loss, acc) in enumerate(zip(loss_history, acc_history), start=1):
        writer.writerow([i, loss, acc])

In [ ]:
# Save the model
model.to('cpu')
model.eval()

with open('/content/drive/MyDrive/GAN_Assignment/C.pkl', 'wb') as f:
    pickle.dump(model, f)


In [ ]:
# Plot the Training Loss and Accuracy

fig, ax1 = plt.subplots()
ax1.plot(range(1, epochs+1), loss_history, 'r-', label='Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='r')

ax2 = ax1.twinx()
ax2.plot(range(1, epochs+1), acc_history, 'b-', label='Accuracy')
ax2.set_ylabel('Accuracy', color='b')

plt.title('Training Loss & Accuracy per Epoch')
plt.show()

In [ ]:
# Load the model for evaluation
with open('/content/drive/MyDrive/GAN_Assignment/C.pkl', 'rb') as f:
    model = pickle.load(f)
model.eval()


In [ ]:
# Model Evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

correct, total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(predicted.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

test_accuracy = correct / total
print(f"Test Accuracy: {test_accuracy:.4f}")
print(classification_report(all_labels, all_preds, digits=4))
print(confusion_matrix(all_labels, all_preds))

In [ ]:
with open('/content/drive/MyDrive/GAN_Assignment/classifier_test_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['metric', 'value'])
    writer.writerow(['test_accuracy', test_accuracy])
    # add S0/S1 classification errors here later once you have them